# Phase B: 將微調後的 FunctionGemma 轉換為 GGUF (INT8)

## v3 版本 - 修正 libllama.so 與 gguf GEMMA4 問題

### v3 修正項目
1. **llama-quantize 改用 static linking** (`-DBUILD_SHARED_LIBS=OFF`)，binary 變獨立單檔，不再需要 `libllama.so`
2. **gguf-py 一起打包到 Drive**，和 convert_hf_to_gguf.py 版本完全匹配，解決 `GEMMA4` AttributeError

### 重要改進 (延續 v2)
**只在「第一次」編譯 llama-quantize (~5-8 分鐘)**，之後都從 Google Drive 載入預編好的工具 (~10 秒)。

### 檔案結構 (存在你的 Google Drive)
```
/content/drive/MyDrive/work/rpi5/
├── llama_cpp_tools/                       ← 持久化的編譯成果
│   ├── llama-quantize                     ← static binary (獨立)
│   ├── convert_hf_to_gguf.py              ← 轉換腳本
│   ├── gguf-py/                           ← Python gguf package (版本匹配)
│   └── BUILD_INFO.txt                     ← 編譯時間戳 / commit hash
├── functiongemma_finetuned/               ← Phase A 的輸出模型
└── functiongemma_gguf/                    ← Phase B 的輸出 .gguf 檔
```

**如果你是從 v2 升級的**: 執行時會自動偵測舊版 (缺 gguf-py)，並觸發重編。

**Runtime**: CPU 即可 (不需要 GPU)

## Step 1: 掛載 Google Drive 並設定路徑

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ===== 路徑設定 =====
PROJECT_ROOT = "/content/drive/MyDrive/work/rpi5"
TOOLS_DIR = f"{PROJECT_ROOT}/llama_cpp_tools"
MODEL_DIR = f"{PROJECT_ROOT}/functiongemma_finetuned"
GGUF_OUTPUT_DIR = f"{PROJECT_ROOT}/functiongemma_gguf"

os.makedirs(TOOLS_DIR, exist_ok=True)
os.makedirs(GGUF_OUTPUT_DIR, exist_ok=True)

print(f"專案根目錄: {PROJECT_ROOT}")
print(f"  工具目錄: {TOOLS_DIR}")
print(f"  模型目錄: {MODEL_DIR}")
print(f"  GGUF 輸出: {GGUF_OUTPUT_DIR}")
print()

if os.path.exists(MODEL_DIR):
    print(f"✅ 微調模型找到了:")
    for f in sorted(os.listdir(MODEL_DIR)):
        size = os.path.getsize(os.path.join(MODEL_DIR, f)) / (1024*1024)
        print(f"     {f} ({size:.1f} MB)")
else:
    print(f"⚠️  找不到微調模型: {MODEL_DIR}")
    print(f"    請確認 Phase A 已完成，或修改 MODEL_DIR 指向你的模型位置")

## Step 2: 檢查 Drive 上是否有 v3 格式的工具

**v3 的工具必須包含 `gguf-py/` 目錄**。如果是舊版 v2 (沒有 gguf-py) 會被標記為不完整，觸發重編。

In [ ]:
TOOLS_QUANTIZE = f"{TOOLS_DIR}/llama-quantize"
TOOLS_CONVERT = f"{TOOLS_DIR}/convert_hf_to_gguf.py"
TOOLS_GGUF_PY = f"{TOOLS_DIR}/gguf-py"
TOOLS_INFO = f"{TOOLS_DIR}/BUILD_INFO.txt"

# v3 判定：三個必要元件都要在
tools_ready = (
    os.path.exists(TOOLS_QUANTIZE) and
    os.path.exists(TOOLS_CONVERT) and
    os.path.isdir(TOOLS_GGUF_PY) and
    os.path.getsize(TOOLS_QUANTIZE) > 100000
)

# 額外檢查：如果 BUILD_INFO 裡沒有 "v3"，視為舊版
if tools_ready and os.path.exists(TOOLS_INFO):
    with open(TOOLS_INFO) as f:
        info_content = f.read()
    if "build_version: v3" not in info_content:
        tools_ready = False
        print("⚠️  偵測到 v2 舊版工具 (缺少 gguf-py 或 static binary)，將觸發重編")

if tools_ready:
    print("✅ 找到 Drive 上 v3 預編好的工具，可以跳過編譯!")
    print()
    if os.path.exists(TOOLS_INFO):
        with open(TOOLS_INFO) as f:
            print("--- 編譯資訊 ---")
            print(f.read())
    print("工具檔案:")
    print(f"  llama-quantize:  {os.path.getsize(TOOLS_QUANTIZE)/1024/1024:.1f} MB (static)")
    print(f"  convert script:  {os.path.getsize(TOOLS_CONVERT)/1024:.1f} KB")
    gguf_py_files = sum(len(files) for _, _, files in os.walk(TOOLS_GGUF_PY))
    print(f"  gguf-py 目錄:    {gguf_py_files} 個檔案")
    print()
    print("→ 可以直接跳到 Step 4 (載入工具到本地)")
else:
    print("❌ Drive 上沒有 v3 預編好的工具")
    print("→ 需要執行 Step 3 進行首次編譯 (~5-8 分鐘)")
    print()
    if os.path.exists(TOOLS_DIR):
        print(f"目前 {TOOLS_DIR} 的內容:")
        for f in os.listdir(TOOLS_DIR):
            fp = os.path.join(TOOLS_DIR, f)
            if os.path.isfile(fp):
                print(f"  {f} ({os.path.getsize(fp)} bytes)")
            else:
                print(f"  {f}/ (目錄)")

## Step 3: 編譯 (只在首次或 v2→v3 升級時執行)

關鍵改動：
- 使用 `-DBUILD_SHARED_LIBS=OFF` 產生 static binary (不需要 libllama.so)
- 把 `gguf-py/` 一起打包到 Drive，確保 Python 端版本匹配

In [ ]:
%%bash
TOOLS="/content/drive/MyDrive/work/rpi5/llama_cpp_tools"

# 若 v3 工具已齊全就跳過
if [ -f "$TOOLS/llama-quantize" ] && [ -f "$TOOLS/convert_hf_to_gguf.py" ] && \
   [ -d "$TOOLS/gguf-py" ] && grep -q "build_version: v3" "$TOOLS/BUILD_INFO.txt" 2>/dev/null; then
    echo "⚡ v3 工具已存在，跳過編譯步驟"
    exit 0
fi

echo "========================================"
echo "  編譯 v3 工具 (~5-8 分鐘)"
echo "  - static linking (獨立 binary)"
echo "  - 打包 gguf-py (避免 gguf 版本不相容)"
echo "========================================"

cd /content

# 若之前有殘留的 build cache，清掉以確保 static linking 生效
if [ -d "llama.cpp" ]; then
    echo "清除舊的 llama.cpp 目錄以確保乾淨編譯..."
    rm -rf llama.cpp
fi

git clone --depth 1 https://github.com/ggml-org/llama.cpp
cd llama.cpp
GIT_COMMIT=$(git rev-parse --short HEAD)
echo "llama.cpp commit: $GIT_COMMIT"
echo ""

# 關鍵：-DBUILD_SHARED_LIBS=OFF 產生 static binary
cmake -B build \
    -DCMAKE_BUILD_TYPE=Release \
    -DBUILD_SHARED_LIBS=OFF \
    -DLLAMA_BUILD_TESTS=OFF \
    -DLLAMA_BUILD_EXAMPLES=OFF \
    -DLLAMA_BUILD_SERVER=OFF 2>&1 | tail -5

echo ""
echo "開始編譯 llama-quantize (static)..."
time cmake --build build --target llama-quantize -j$(nproc)

# 驗證產出的是 static binary (不該 depend 任何 libllama.so)
echo ""
echo "--- 驗證 binary 依賴 ---"
ldd build/bin/llama-quantize | grep -E '(libllama|libggml)' && \
    echo "⚠️  警告：binary 仍有 llama/ggml 動態連結!" || \
    echo "✅ 確認為 static binary (沒有 libllama/libggml 依賴)"

# --- 複製到 Drive ---
echo ""
echo "========================================"
echo "  複製工具到 Google Drive"
echo "========================================"

mkdir -p "$TOOLS"

# 清除舊檔 (避免殘留 v2 殘件)
rm -rf "$TOOLS/gguf-py" "$TOOLS/llama-quantize" "$TOOLS/convert_hf_to_gguf.py"

# 1. binary
cp build/bin/llama-quantize "$TOOLS/"
chmod +x "$TOOLS/llama-quantize"

# 2. 轉換 script
cp convert_hf_to_gguf.py "$TOOLS/"

# 3. gguf-py 整個目錄 (版本要和 convert script 完全匹配)
cp -r gguf-py "$TOOLS/"

# 4. 編譯資訊
cat > "$TOOLS/BUILD_INFO.txt" << EOF
build_version: v3
編譯時間: $(date)
llama.cpp commit: $GIT_COMMIT
Python 版本: $(python3 --version)
編譯平台: $(uname -a)
編譯選項: -DBUILD_SHARED_LIBS=OFF (static binary)
EOF

echo ""
echo "✅ 已存到 Google Drive:"
ls -la "$TOOLS/"
echo ""
echo "gguf-py 內容 (前 10 個):"
ls "$TOOLS/gguf-py/gguf/" | head -10
echo ""
echo "下次執行此 notebook 時，會直接從 Drive 載入，不再需要編譯!"

## Step 4: 載入工具到本地環境

關鍵：把 **Drive 的 gguf-py 加到 PYTHONPATH**，而不是 pip install gguf。這樣確保 Python 端的 gguf package 和 convert_hf_to_gguf.py 版本完全一致。

In [ ]:
import shutil
import sys

LOCAL_TOOLS = "/content/tools"

# 清掉舊的 (避免 v2 殘留)
if os.path.exists(LOCAL_TOOLS):
    shutil.rmtree(LOCAL_TOOLS)
os.makedirs(LOCAL_TOOLS, exist_ok=True)

local_quantize = f"{LOCAL_TOOLS}/llama-quantize"
local_convert = f"{LOCAL_TOOLS}/convert_hf_to_gguf.py"
local_gguf_py = f"{LOCAL_TOOLS}/gguf-py"

shutil.copy2(TOOLS_QUANTIZE, local_quantize)
shutil.copy2(TOOLS_CONVERT, local_convert)
shutil.copytree(TOOLS_GGUF_PY, local_gguf_py)
os.chmod(local_quantize, 0o755)

print(f"✅ 工具已載入本地:")
print(f"   {local_quantize}")
print(f"   {local_convert}")
print(f"   {local_gguf_py}/")

# 同時為本 notebook 自己的 Python 環境加入 gguf-py 路徑
# (因為 Step 7 會用 GGUFReader 驗證)
if local_gguf_py not in sys.path:
    sys.path.insert(0, local_gguf_py)

# 移除可能衝突的 pip 版 gguf
try:
    import gguf
    if 'site-packages' in gguf.__file__:
        print(f"\n⚠️  偵測到 pip 版 gguf: {gguf.__file__}")
        print("    將嘗試解除安裝以避免衝突...")
        !pip uninstall -y gguf 2>&1 | tail -2
        # 重載
        import importlib
        if 'gguf' in sys.modules:
            del sys.modules['gguf']
except ImportError:
    pass

In [ ]:
# 安裝 convert_hf_to_gguf.py 其他需要的依賴 (不包含 gguf)
!pip install -q sentencepiece protobuf safetensors
!pip install -q "transformers>=4.50.0"

print("\n✅ Python 依賴安裝完成")

In [ ]:
# --- 驗證 1: llama-quantize 是 static binary，可獨立執行 ---
print("=== 驗證 llama-quantize ===")
!{local_quantize} --help 2>&1 | head -5

# 檢查動態函式庫依賴
print("\n--- 動態函式庫依賴檢查 (不應看到 libllama.so) ---")
!ldd {local_quantize} | grep -E '(libllama|libggml)' && echo '⚠️  仍有動態依賴!' || echo '✅ 確認無 libllama/libggml 動態依賴'

# --- 驗證 2: convert_hf_to_gguf.py 可載入 (無 GEMMA4 錯誤) ---
print("\n=== 驗證 convert_hf_to_gguf.py ===")
# 設定 PYTHONPATH 讓 script 能找到匹配版本的 gguf
import subprocess
env = os.environ.copy()
env['PYTHONPATH'] = local_gguf_py + ':' + env.get('PYTHONPATH', '')

result = subprocess.run(
    ['python3', local_convert, '--help'],
    env=env, capture_output=True, text=True
)
print(result.stdout[:500] if result.returncode == 0 else f"❌ 錯誤:\n{result.stderr[-800:]}")

## Step 5: 轉換 SafeTensors → GGUF F16

注意：執行 convert script 時要設定 `PYTHONPATH` 指向 `gguf-py` 目錄。

In [ ]:
GGUF_F16_NAME = "functiongemma-270m-finetuned-f16.gguf"
GGUF_Q8_NAME = "functiongemma-270m-finetuned-q8_0.gguf"

local_f16 = f"/content/{GGUF_F16_NAME}"
local_q8 = f"/content/{GGUF_Q8_NAME}"

print(f"模型來源: {MODEL_DIR}")
print(f"F16 輸出: {local_f16}")

In [ ]:
# 注意：PYTHONPATH=... 讓 Python 先找到打包的 gguf-py 版本
!PYTHONPATH={local_gguf_py} python3 {local_convert} "{MODEL_DIR}" \
    --outtype f16 \
    --outfile "{local_f16}"

if os.path.exists(local_f16):
    size_mb = os.path.getsize(local_f16) / (1024*1024)
    print(f"\n✅ F16 GGUF 產生成功: {size_mb:.1f} MB")
else:
    print("\n❌ 轉換失敗，請檢查上方錯誤訊息")

## Step 6: 量化 F16 → Q8_0 (INT8)

Static binary 可以獨立執行，不再需要 libllama.so。

In [ ]:
!{local_quantize} "{local_f16}" "{local_q8}" Q8_0

if os.path.exists(local_q8):
    f16_size = os.path.getsize(local_f16) / (1024*1024)
    q8_size = os.path.getsize(local_q8) / (1024*1024)
    ratio = q8_size / f16_size * 100
    print(f"\n✅ 量化完成!")
    print(f"   F16:  {f16_size:.1f} MB")
    print(f"   Q8_0: {q8_size:.1f} MB ({ratio:.0f}% of F16)")
    print(f"   節省: {f16_size - q8_size:.1f} MB")
else:
    print("\n❌ 量化失敗")

### (選擇性) 其他量化版本

In [ ]:
# # Q4_K_M / Q6_K (取消註解即可使用)
# local_q4 = f"/content/functiongemma-270m-finetuned-q4_k_m.gguf"
# !{local_quantize} "{local_f16}" "{local_q4}" Q4_K_M
# local_q6 = f"/content/functiongemma-270m-finetuned-q6_k.gguf"
# !{local_quantize} "{local_f16}" "{local_q6}" Q6_K

## Step 7: 驗證 GGUF 檔案

In [ ]:
# 確保用的是打包的 gguf-py 版本 (不是 pip 的)
import sys
if local_gguf_py not in sys.path:
    sys.path.insert(0, local_gguf_py)

# 移除快取的 gguf module，強制重新 import
for mod in list(sys.modules.keys()):
    if mod.startswith('gguf'):
        del sys.modules[mod]

from gguf import GGUFReader

reader = GGUFReader(local_q8)

print(f"GGUF 檔案: {local_q8}")
print(f"\n元資料:")
for key, field in list(reader.fields.items())[:15]:
    if key.startswith('general.'):
        try:
            val = field.parts[-1].tolist()
            if isinstance(val, list) and len(val) == 1:
                val = val[0]
            if isinstance(val, bytes):
                val = val.decode('utf-8')
            print(f"  {key}: {val}")
        except Exception:
            pass

print(f"\nTensor 數量: {len(reader.tensors)}")
for t in reader.tensors[:3]:
    print(f"  {t.name}: shape={t.shape}, type={t.tensor_type.name}")

## Step 8: 將最終 GGUF 檔複製到 Google Drive

In [ ]:
final_q8_path = f"{GGUF_OUTPUT_DIR}/{GGUF_Q8_NAME}"

print(f"複製到 Drive 中... (~{os.path.getsize(local_q8)/1024/1024:.0f} MB)")
shutil.copy2(local_q8, final_q8_path)

print(f"\n✅ 完成! 檔案位置:")
print(f"   {final_q8_path}")
print(f"\nGGUF 目錄內容:")
for f in os.listdir(GGUF_OUTPUT_DIR):
    size = os.path.getsize(os.path.join(GGUF_OUTPUT_DIR, f)) / (1024*1024)
    print(f"   {f} ({size:.1f} MB)")

## Step 9: 傳輸到 RPI5

In [ ]:
print("=" * 60)
print("  傳輸方式")
print("=" * 60)
print()
print(f"檔案位置: {final_q8_path}")
print(f"檔案大小: {os.path.getsize(final_q8_path)/1024/1024:.1f} MB")
print()
print("方法 1: 從 Colab 下載到 PC")
print(f"  from google.colab import files")
print(f"  files.download('{final_q8_path}')")
print()
print("方法 2: PC 從 Google Drive 同步後 scp 到 RPI5")
print(f"  scp {GGUF_Q8_NAME} pi@<RPI5_IP>:~/models/")
print()
print("方法 3: RPI5 用 rclone 直接拉 (見 RPI5_指令速查.txt [3])")
print(f"  rclone copy gdrive:work/rpi5/functiongemma_gguf/ ~/models/")

---

## 附錄：強制重新編譯

例如 llama.cpp 有大更新時，或工具有問題時，刪除 Drive 上的工具資料夾：

In [ ]:
# # ⚠️ 只有想重新編譯時才執行
# import shutil
# if os.path.exists(TOOLS_DIR):
#     shutil.rmtree(TOOLS_DIR)
#     print(f"已刪除 {TOOLS_DIR}")
#     print("重新執行 Step 2~3 會觸發重新編譯")